# OOM trial for SagiFBT

This notebook runs a single stress trial of `sagifbt` using the **maximum** `xgb_n_estimators` value from `tests/sagifbt_run_new.py` (`1000`) and random choices for other hyperparameters.

Goal: check whether the run goes out-of-memory (OOM).

In [1]:
import os
import sys
import time
import random
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
from xgboost import XGBClassifier

# Optional: memory diagnostics (works on macOS/Linux)
try:
    import resource
except Exception:
    resource = None

# Ensure local imports resolve from repo root
if os.getcwd().endswith('/tests'):
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
else:
    repo_root = os.getcwd()

if repo_root not in sys.path:
    sys.path.append(repo_root)

from FBT import FBT
from tests.data_utils import DataFactory_clf

print('Repo root:', repo_root)
print('Start time:', datetime.now().isoformat(timespec='seconds'))

Repo root: /Users/beliz/Desktop/thesis_project/SagiXGBoostTreeApproximator
Start time: 2026-03-24T14:57:00


In [2]:
# Settings (tweak if needed)
dataset = 'room'
fold = 0
seed = 12345
use_device = 'cpu'   # set to 'cuda' if you want GPU test
n_jobs = -1

rng = np.random.default_rng(seed)
random.seed(seed)

# Match the grid from tests/sagifbt_run_new.py, but force max estimators
hyperparams = {
    'xgb_n_estimators': 1000,  # max from grid [50,100,250,500,750,1000]
    'xgb_max_depth': int(rng.integers(3, 9)),
    'xgb_learning_rate': float(rng.choice([0.01, 0.05, 0.1, 0.2, 0.3])),
    'xgb_colsample_bytree': float(rng.choice([0.25, 0.5, 0.75, 1.0])),
    'xgb_subsample': float(rng.choice([0.5, 0.63, 0.8, 1.0])),
    'xgb_min_child_weight': int(rng.choice([1, 3, 5, 10])),
    'xgb_reg_lambda': float(rng.choice([0.0, 0.1, 1.0, 5.0, 10.0])),
    'xgb_reg_alpha': float(rng.choice([0.0, 0.1, 0.5, 1.0])),
    'xgb_gamma': float(rng.choice([0.0, 0.1, 0.5, 1.0, 5.0])),
    'max_depth': int(rng.integers(2, 7)),
    'max_number_of_conjunctions': int(rng.choice([100, 250, 500, 1000])),
    'min_forest_size': int(rng.choice([1, 2, 5])),
}

print('Sampled hyperparameters:')
for k, v in hyperparams.items():
    print(f'  {k}: {v}')

# Load data
data_factory = DataFactory_clf(dataset, cache_dir=os.path.join(repo_root, 'data'))
X_train, y_train, X_val, y_val, X_test, y_test = data_factory.get_data(fold)

feature_cols = [f'f{i}' for i in range(X_train.shape[1])]
label_col = 'label'

train_df = pd.DataFrame(X_train, columns=feature_cols)
train_df[label_col] = y_train

# Build and run
xgb_model = XGBClassifier(
    n_estimators=hyperparams['xgb_n_estimators'],
    max_depth=hyperparams['xgb_max_depth'],
    learning_rate=hyperparams['xgb_learning_rate'],
    colsample_bytree=hyperparams['xgb_colsample_bytree'],
    subsample=hyperparams['xgb_subsample'],
    min_child_weight=hyperparams['xgb_min_child_weight'],
    reg_lambda=hyperparams['xgb_reg_lambda'],
    reg_alpha=hyperparams['xgb_reg_alpha'],
    gamma=hyperparams['xgb_gamma'],
    random_state=seed,
    nthread=n_jobs,
    device=use_device,
    eval_metric='mlogloss',
    verbosity=1,
)

clf = FBT(
    max_depth=hyperparams['max_depth'],
    min_forest_size=hyperparams['min_forest_size'],
    max_number_of_conjunctions=hyperparams['max_number_of_conjunctions'],
)

start = time.time()
print('\nFitting XGBoost...')
try:
    xgb_model.fit(X_train, y_train)
    print('Fitting FBT...')
    clf.fit(
        train=train_df,
        feature_cols=feature_cols,
        label_col=label_col,
        xgb_model=xgb_model,
    )
    status = 'success'
except MemoryError:
    status = 'memory_error'
    print('MemoryError raised during fit.')
except Exception as e:
    status = f'error: {type(e).__name__}'
    print('Exception during fit:')
    traceback.print_exc()

elapsed = time.time() - start
print(f'\nStatus: {status}')
print(f'Elapsed: {elapsed:.2f}s')

if resource is not None:
    # On macOS ru_maxrss is bytes; on Linux it is KB. This still gives a useful peak signal.
    print('Peak RSS (platform-dependent units):', resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)

Sampled hyperparameters:
  xgb_n_estimators: 1000
  xgb_max_depth: 7
  xgb_learning_rate: 0.05
  xgb_colsample_bytree: 1.0
  xgb_subsample: 0.63
  xgb_min_child_weight: 1
  xgb_reg_lambda: 5.0
  xgb_reg_alpha: 0.5
  xgb_gamma: 1.0
  max_depth: 6
  max_number_of_conjunctions: 250
  min_forest_size: 5

Fitting XGBoost...
Fitting FBT...
Start pruning
Create conjunction set from training data instances
Number of conjunctions created from data: 1732
Create complete conjunction set
Size at iteration 2: 251
Size at iteration 3: 251
Size at iteration 4: 251
Size at iteration 5: 251
Size at iteration 6: 251
Size at iteration 7: 251
Size at iteration 8: 251
Size at iteration 9: 251
Size at iteration 10: 251
Size at iteration 11: 251
Size at iteration 12: 251
Size at iteration 13: 251
Size at iteration 14: 251
Size at iteration 15: 251
Size at iteration 16: 251
Size at iteration 17: 251
Size at iteration 18: 251
Size at iteration 19: 251
Size at iteration 20: 251
Size at iteration 21: 251
Size at

In [3]:
# Multi-trial OOM loop: run cell 1 (imports) first. Skip cell 2 if you only want this loop.
# Forces xgb_n_estimators=1000 each time, resamples other HPs. Loads data once below.
# Set max_trials=None to run until MemoryError (kernel may die before that on macOS).

def sample_hyperparams(rng, force_xgb_n_estimators=1000):
    return {
        "xgb_n_estimators": force_xgb_n_estimators,
        "xgb_max_depth": int(rng.integers(3, 9)),
        "xgb_learning_rate": float(rng.choice([0.01, 0.05, 0.1, 0.2, 0.3])),
        "xgb_colsample_bytree": float(rng.choice([0.25, 0.5, 0.75, 1.0])),
        "xgb_subsample": float(rng.choice([0.5, 0.63, 0.8, 1.0])),
        "xgb_min_child_weight": int(rng.choice([1, 3, 5, 10])),
        "xgb_reg_lambda": float(rng.choice([0.0, 0.1, 1.0, 5.0, 10.0])),
        "xgb_reg_alpha": float(rng.choice([0.0, 0.1, 0.5, 1.0])),
        "xgb_gamma": float(rng.choice([0.0, 0.1, 0.5, 1.0, 5.0])),
        "max_depth": int(rng.integers(2, 7)),
        "max_number_of_conjunctions": int(rng.choice([100, 250, 500, 1000])),
        "min_forest_size": int(rng.choice([1, 2, 5])),
    }

dataset = "room"
fold = 0
base_seed = 12345
use_device = "cpu"
n_jobs = -1
max_trials = 50  # None = run until MemoryError

data_factory = DataFactory_clf(dataset, cache_dir=os.path.join(repo_root, "data"))
X_train, y_train, X_val, y_val, X_test, y_test = data_factory.get_data(fold)

feature_cols = [f"f{i}" for i in range(X_train.shape[1])]
label_col = "label"

train_df = pd.DataFrame(X_train, columns=feature_cols)
train_df[label_col] = y_train

trial = 0
while True:
    if max_trials is not None and trial >= max_trials:
        print(f"Stopped after {max_trials} trials (no MemoryError in Python).")
        break

    seed = base_seed + trial
    rng = np.random.default_rng(seed)
    hp = sample_hyperparams(rng, force_xgb_n_estimators=1000)

    print(f"\n=== trial {trial} | seed {seed} ===")
    for k, v in hp.items():
        print(f"  {k}: {v}")

    xgb_model = XGBClassifier(
        n_estimators=hp["xgb_n_estimators"],
        max_depth=hp["xgb_max_depth"],
        learning_rate=hp["xgb_learning_rate"],
        colsample_bytree=hp["xgb_colsample_bytree"],
        subsample=hp["xgb_subsample"],
        min_child_weight=hp["xgb_min_child_weight"],
        reg_lambda=hp["xgb_reg_lambda"],
        reg_alpha=hp["xgb_reg_alpha"],
        gamma=hp["xgb_gamma"],
        random_state=seed,
        nthread=n_jobs,
        device=use_device,
        eval_metric="mlogloss",
        verbosity=0,
    )

    clf = FBT(
        max_depth=hp["max_depth"],
        min_forest_size=hp["min_forest_size"],
        max_number_of_conjunctions=hp["max_number_of_conjunctions"],
    )

    t0 = time.time()
    try:
        xgb_model.fit(X_train, y_train)
        clf.fit(
            train=train_df,
            feature_cols=feature_cols,
            label_col=label_col,
            xgb_model=xgb_model,
        )
    except MemoryError:
        print(f"MemoryError on trial {trial} after {time.time() - t0:.2f}s — stopping.")
        break
    except Exception:
        print(f"Other exception on trial {trial} after {time.time() - t0:.2f}s:")
        traceback.print_exc()
    else:
        print(f"OK in {time.time() - t0:.2f}s")
        if resource is not None:
            print(
                "Peak RSS (platform-dependent units):",
                resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            )

    trial += 1


=== trial 0 | seed 12345 ===
  xgb_n_estimators: 1000
  xgb_max_depth: 7
  xgb_learning_rate: 0.05
  xgb_colsample_bytree: 1.0
  xgb_subsample: 0.63
  xgb_min_child_weight: 1
  xgb_reg_lambda: 5.0
  xgb_reg_alpha: 0.5
  xgb_gamma: 1.0
  max_depth: 6
  max_number_of_conjunctions: 250
  min_forest_size: 5
Start pruning
Create conjunction set from training data instances
Number of conjunctions created from data: 1732
Create complete conjunction set
Size at iteration 2: 251
Size at iteration 3: 251
Size at iteration 4: 251
Size at iteration 5: 251
Size at iteration 6: 251
Size at iteration 7: 251
Size at iteration 8: 251
Size at iteration 9: 251
Size at iteration 10: 251
Size at iteration 11: 251
Size at iteration 12: 251
Size at iteration 13: 251
Size at iteration 14: 251
Size at iteration 15: 251
Size at iteration 16: 251
Size at iteration 17: 251
Size at iteration 18: 251
Size at iteration 19: 251
Size at iteration 20: 251
Size at iteration 21: 251
Size at iteration 22: 251
Size at ite

KeyboardInterrupt: 